# Autogen

Microsoft's agent framework
- Designed for multi-agent conversation
- Patterns of agent interaction
- Supports tool calling, code execution

Other agent frameworks: CrewAI, Swarm, LangGraph

**Software versions:** Python 3.10+, Autogen 0.4.0+

## Setup

Autogen install

In [ ]:
!pip install autogen

Installing LLM clients, you can use many LLM providers:
- OpenAI
- Azure
- Claude
- Gemini
- Ollama

We will be using Ollama to run open-source models locally.

In [ ]:
!pip install "autogen-ext[ollama]"

In [ ]:
!pip install "autogen-ext[openai]"

In [ ]:
# chat client - openai
from autogen_ext.models.openai import OpenAIChatCompletionClient

openai_client = OpenAIChatCompletionClient(
    model="gpt-4o",
    api_key="OPENAI_API_KEY"
)

If you use an API key, make sure its secure! Don't put it in code and accidentally leak it, use environment variables, for example:

In [ ]:
# export OPENAI_API_KEY=MY_API_KEY

import os
api_key = os.getenv("OPENAI_API_KEY")

In [6]:
# ollama client
from autogen_ext.models.ollama import OllamaChatCompletionClient

ollama_client = OllamaChatCompletionClient(
    model="mistral:latest",
    # host="http://localhost:11434/v1",
    seed=42
)

## Autogen example
Simple Coding example with a coder and reviewer

In [17]:
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent

In [18]:
#create some agents
coder = AssistantAgent(
    name="Coder",
    model_client=ollama_client,
    system_message=("You are a helpful Python programmer. Write short Python functions to solve the given task.")     
)

reviewer = AssistantAgent(
    name="Reviewer",
    model_client=ollama_client,
    system_message=(
        "You are a strict code reviewer. Inspect the Coder's output, find bugs or styling issues and fix them."
        "Output the reviewed version with comments on the changes made."
        "Let the Coder make further changes to the reviewed version."
        "Only output 'TASKDONE' when you are fully satisfied of the code's quality."
    )
)

In [19]:
# set a termination condition
from autogen_agentchat.conditions import TextMentionTermination

termination = TextMentionTermination("TASKDONE")

There are many kinds of termination messages (https://microsoft.github.io/autogen/stable//user-guide/agentchat-user-guide/tutorial/termination.html)

- Max messages
- Text mention
- Token usage
- Timeout (duration in seconds)
- Source match - after a particular agent responds
- Function call - when a tool call is executed with the matching name

Text mention termination can sometimes be unreliable, but provides models with a means to end the chat when the task is complete.

In [14]:
# setup the group chat
from autogen_agentchat.teams import RoundRobinGroupChat

groupchat = RoundRobinGroupChat(
    [coder, reviewer], # list of agents
    termination_condition=termination,
    max_turns=5
)

NameError: name 'coder' is not defined

There are a few different group chat "patterns":
- **Round Robin**: participants take turns one by one
- **Selector**: one agent acts as a selector to choose the next speaker based on the context and conversation so far
- **Swarm**: selects the next speaker based on handoff messages from one agent to another, effectively allowing the current acting agent to decide who should speak next.

### Run the task

In [97]:
from autogen_agentchat.ui import Console

task_description = "Write a Python function that returns the factorial of a number using recursion."

await Console(groupchat.run_stream(task=task_description))

---------- TextMessage (user) ----------
Write a Python function that returns the factorial of a number using recursion.
---------- TextMessage (Coder) ----------
 Sure! Here's a simple Python function that calculates the factorial of a number using recursion:

```python
def factorial(n):
    if n == 0 or n == 1:
        return 1
    else:
        return n * factorial(n-1)
```

You can use this function like this:

```python
print(factorial(5))  # Output: 120
```

This code works by calling the `factorial()` function recursively with a decreasing value of `n` until it reaches either 0 or 1, at which point it starts returning results. The product of all these returned values is the factorial of the input number.
---------- TextMessage (Reviewer) ----------
 Here's a slightly improved version of your code:

```python
def factorial(n):
    """Calculate the factorial of a number using recursion."""
    if n <= 0:
        raise ValueError("Factorial only defined for positive integers.")
   

TaskResult(messages=[TextMessage(id='6f5a1680-d747-40a3-b525-590e047f2ea7', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 10, 28, 11, 39, 48, 341937, tzinfo=datetime.timezone.utc), content='Write a Python function that returns the factorial of a number using recursion.', type='TextMessage'), TextMessage(id='cadd2377-2fb9-4217-ac05-a9a736282361', source='Coder', models_usage=RequestUsage(prompt_tokens=41, completion_tokens=165), metadata={}, created_at=datetime.datetime(2025, 10, 28, 11, 39, 55, 442197, tzinfo=datetime.timezone.utc), content=" Sure! Here's a simple Python function that calculates the factorial of a number using recursion:\n\n```python\ndef factorial(n):\n    if n == 0 or n == 1:\n        return 1\n    else:\n        return n * factorial(n-1)\n```\n\nYou can use this function like this:\n\n```python\nprint(factorial(5))  # Output: 120\n```\n\nThis code works by calling the `factorial()` function recursively with a decreasing value of `

### Multi-Agent Collaboration with Faulty Agents (Huang et al. ICML 2025)

Let's try something more complex with more agents

In [20]:
# Lead coder: writes clean code
lead_coder = AssistantAgent(
    name="LeadCoder",
    model_client=ollama_client,
    system_message=(
        "You are the lead coder. Write clean, correct Python code for the given task. "
    )
)

# Faulty coder, intentionally injects bugs
faulty_coder = AssistantAgent(
    name="FaultyCoder",
    model_client=ollama_client,
    system_message=(
        "You are a secondary coder who tries to 'improve' code. "
        "Sometimes you make small errors (e.g., wrong variable names, missing returns, poor styling choices). "
        "Do not mention you're faulty. Put the comment ### above faulty lines you introduce. Just modify and send your version."
    )
)

# Reviewer, finds and fixes issues
reviewer = AssistantAgent(
    name="Reviewer",
    model_client=ollama_client,
    system_message=(
        "You are a code reviewer. Review the last code, find logical or stylistic issues, fix them, "
        "and output corrected code. End with 'READYINSPECTION' when satisfied."
    )
)

# Inspector, validates the final code
inspector = AssistantAgent(
    name="Inspector",
    model_client=ollama_client,
    system_message=(
        "You are the inspector. Given the final code and the original task, "
        "simulate running it mentally. If correct and readable, output 'TASKDONE'. "
        "Otherwise, request further fixes."
    )
)

user_proxy = UserProxyAgent("User")

Note all the system messages. Each agent needs its own description on what its role is. These system messages can get much more detailed and complicated if needed, and is a easy first step to customise the agent system that you want.

In [106]:
termination = TextMentionTermination("TASKDONE")

chat = RoundRobinGroupChat(
    [lead_coder, faulty_coder, reviewer, inspector],
    termination_condition=termination,
    max_turns=8
)

In [108]:
await Console(chat.run_stream(task="Build a modular data processing pipeline for analyzing customer reviews."))

---------- TextMessage (user) ----------
Build a modular data processing pipeline for analyzing customer reviews.
---------- TextMessage (LeadCoder) ----------
 To build a modular data processing pipeline for analyzing customer reviews, I'll create several functions that perform different tasks and then combine them in an easy-to-use pipeline. Here's the structure of the code:

```python
import re
from collections import defaultdict
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize

def load_data(filepath):
    """Load customer reviews from a file"""
    with open(filepath, 'r') as f:
        data = [line.strip() for line in f]
    return data

def preprocess_reviews(reviews):
    """Preprocess the customer reviews - lowercasing, tokenizing, removing stopwords, and lemmatizing"""
    review_tokens = []
    nltk_stopwords = set(stopwords.words("english"))
    tokenizer = word_tokenize

    for review in reviews:
        tokens = tokenizer(review)
 

TaskResult(messages=[TextMessage(id='17cfd703-50a9-4591-a41d-f071a627f82c', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 10, 28, 12, 22, 28, 188583, tzinfo=datetime.timezone.utc), content='Build a modular data processing pipeline for analyzing customer reviews.', type='TextMessage'), TextMessage(id='bdf303ee-8f00-4739-9acc-ce005fbdb487', source='LeadCoder', models_usage=RequestUsage(prompt_tokens=39, completion_tokens=678), metadata={}, created_at=datetime.datetime(2025, 10, 28, 12, 22, 55, 191488, tzinfo=datetime.timezone.utc), content=' To build a modular data processing pipeline for analyzing customer reviews, I\'ll create several functions that perform different tasks and then combine them in an easy-to-use pipeline. Here\'s the structure of the code:\n\n```python\nimport re\nfrom collections import defaultdict\nfrom nltk.corpus import stopwords\nfrom nltk.tokenize import word_tokenize, sent_tokenize\n\ndef load_data(filepath):\n    """Load cust

# Customising Autogen - Intrinsic Memory Agents
The BaseChatAgent can be extended to a custom class, so we can add any additional features we want to use.

We need to fill in some abstract functions:
- on_messages(): what to do when getting incoming messages, returns a Response
- on_reset(): reset agent to initial state (clearing memory, cache, etc.)
- produced_message_types(): what kind of messages can be returned in a Response

In [111]:
from autogen_agentchat.agents import BaseChatAgent
from autogen_agentchat.messages import TextMessage, UserMessage
from autogen_core.model_context import UnboundedChatCompletionContext
from autogen_core.models import AssistantMessage
from autogen_agentchat.base import Response

class MemoryAgent(BaseChatAgent):
    
    def __init__(self, name, model_client, system_message, description):
        super().__init__(name=name, description=description)
        
        self._model_client = model_client
        self._conversation_history = UnboundedChatCompletionContext()
        self._memory_summary = ""
        self._initial_system_message = system_message
        self._current_system_message = system_message

    @property
    def produced_message_types(self):
        return (TextMessage,)

    async def on_reset(self, cancellation_token):
        self._conversation_history.clear()
        self._memory_summary = ""
        self._current_system_message = self._initial_system_message

    async def on_messages(self, messages, cancellation_token):
        # 1. Store messages in conversation history
        for msg in messages:
            await self._conversation_history.add_message(msg.to_model_message())

        # 2. Memory update
        await self._update_memory_summary()
        
        # 3. Build the context + memory
        model_messages = [
            msg.to_model_message() for msg in messages
        ]
        model_messages.append(
            UserMessage(content=self._current_system_message, source="system")
        )
        
        # 4. Call the model client for inference
        response = await self._model_client.create(model_messages)

        print(response)
        # get usage data
        # usage = RequestUsage(
        #     prompt_tokens=response.usage_metadata.prompt_token_count,
        #     completion_tokens=response.usage_metadata.candidates_token_count,
        # )

        await self._conversation_history.add_message(AssistantMessage(content=response.content, source=self.name))

        # 5. Yield the final response
        return Response(
            chat_message=TextMessage(content=response.content, source=self.name),# models_usage=usage),
            inner_messages=[],
        )


    async def _update_memory_summary(self):
        history_messages = await self._conversation_history.get_messages()

        if len(history_messages) == 0:
            return
        
        # get conversation history
        history_text = "\n".join([
            f"[{msg.source}]: {msg.content if isinstance(msg.content, str) else str(msg.content)}"
            for msg in history_messages
            if hasattr(msg, 'content') and hasattr(msg, 'source')
        ])

        if not history_text:
            return

        # memory update prompt
        summarization_messages = [
            UserMessage(
                content="""Use the entire history of the conversation to 
                populate and update your current memory with factual information.
                Create a concise summary of the key points, decisions, and context from the conversation. 
                Focus on information that would be useful for future reference.""",
                source="system"
            ),
            UserMessage(
                content=f"Summarize the conversation history:\n\n{history_text}",
                source="user"
            )
        ]
        
        try:
            summary_result = await self._model_client.create(
                summarization_messages,
            )
            
            self._memory_summary = summary_result.content
            
            # update system message with memory
            self._update_system_message()
            
        except Exception as e:
            print(f"Failed to update memory summary: {e}")
    
    def _update_system_message(self):
        if self._memory_summary:
            self._current_system_message = (
                f"{self._initial_system_message}\n\n"
                f"MEMORY FROM PREVIOUS CONVERSATIONS:\n{self._memory_summary}"
            )

In [112]:
memory_agent = MemoryAgent(
    name="Inspector",
    model_client=ollama_client,
    system_message=(
        "You are the inspector. Given the final code and the original task, "
        "simulate running it mentally. If correct and readable, output 'TASKDONE'. "
        "Otherwise, request further fixes."
    ),
    description="Inspector with memory."
)

In [113]:
chat = RoundRobinGroupChat(
    [memory_agent, lead_coder],
    termination_condition=termination,
    max_turns=8
)

In [114]:
from autogen_agentchat.ui import Console

await Console(chat.run_stream(task="Build a modular data processing pipeline for analyzing customer reviews."))

---------- TextMessage (user) ----------
Build a modular data processing pipeline for analyzing customer reviews.
finish_reason='stop' content=' In my mental simulation, I can see a modular data processing pipeline in Python, with each component as a separate module:\n\n1. **Data Collection Module** - This module gathers customer review data from various sources such as websites, social media platforms, and feedback forms using APIs or web scraping libraries like BeautifulSoup or Scrapy.\n\n2. **Data Cleaning Module** - The cleaned data is then passed on to this module where irrelevant information is removed, missing values are filled, and the data is prepared for analysis. This can be achieved using data cleaning libraries such as Pandas.\n\n3. **Feature Extraction Module** - This module identifies essential features in the data that can be used to categorize or quantify sentiments in the reviews. Libraries like NLTK can help with tasks like tokenization, stemming, and part-of-speech 

TaskResult(messages=[TextMessage(id='8c288b39-a29f-49f2-ad6e-d626150d9688', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 11, 2, 13, 44, 10, 438864, tzinfo=datetime.timezone.utc), content='Build a modular data processing pipeline for analyzing customer reviews.', type='TextMessage'), TextMessage(id='f198f4c7-7e76-49c6-8494-2d9b6bacb37d', source='Inspector', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 11, 2, 13, 44, 41, 544006, tzinfo=datetime.timezone.utc), content=' In my mental simulation, I can see a modular data processing pipeline in Python, with each component as a separate module:\n\n1. **Data Collection Module** - This module gathers customer review data from various sources such as websites, social media platforms, and feedback forms using APIs or web scraping libraries like BeautifulSoup or Scrapy.\n\n2. **Data Cleaning Module** - The cleaned data is then passed on to this module where irrelevant information is remove

# Customising Autogen - Group Chat

Find more details and further advanced tutorials in https://microsoft.github.io/autogen/stable//user-guide/agentchat-user-guide/tutorial/index.html